<img src="https://raw.githubusercontent.com/Brainchip-Inc/brainchip_devhub/main/docs/assets/0.-BC-dev-hub-LOGO-flicker.svg" alt="BrainChip Dev Hub" width="200"/>

# Hand Gesture Recognition (Jester) — Akida 2 Evaluation

Run Time: ~10 minutes

This notebook takes BrainChip's **TENN spatiotemporal** model for Jester from its float
form to an Akida 2 model, and measures what each step costs in accuracy:

1. evaluate the float model, which sees each clip whole,
2. quantize it to 8 bits with post-training quantization (PTQ), as BrainChip did, and check
   the result against the published 8-bit model,
3. evaluate the quantized model, which is now a **streaming** model: one frame per
   inference,
4. convert it to Akida, evaluate it, and check how it maps onto Akida 2 devices.

There is no training here: the float model is BrainChip's published one (see 'Training
recipe' in the README).

The data is the **BrainChip Jester subset**, one shard of the Jester validation split
(541 clips), downloaded from BrainChip's dataset mirror on first use. Every accuracy in
this notebook is on that subset, not on the full validation split.

In [1]:
# Colab-only setup. Local users: ignore this cell - it does nothing for you.
import sys
if 'google.colab' in sys.modules:
    !wget -q https://raw.githubusercontent.com/Brainchip-Inc/brainchip_devhub/main/akida2/model_zoo/jester/colab_setup.py
import colab_setup
colab_setup.setup()

Not running on Colab — nothing to do. (This step only matters for Colab; local runs already have everything they need.)


## Configuration

`DATA_PATH = None` uses the BrainChip Jester subset. To evaluate on the full validation
split, point it at a Jester directory in the layout described under 'Dataset' in the README.

In [2]:
import numpy as np

DATA_PATH = None     # None: the BrainChip Jester subset, downloaded into data/
BATCH_SIZE = 32      # clips per batch for the Keras models

## Float model

The published float model is built from `Conv3D` layers. It takes a whole clip, 16 frames
of 100 × 100 RGB, and returns a prediction for every frame. Its blocks alternate a
**temporal** convolution (kernel 5 over time, 1 × 1 in space) and a **spatial** one (3 × 3,
1 over time). The temporal convolutions are padded with 4 frames before the clip and none
after, so they are causal: a frame's prediction only depends on the frames up to it.

In [3]:
from akida_models.model_io import load_model
from jester_model import model_path, verify_model

float_path = model_path('float')
verify_model(float_path)   # checks the published sha256 (and that Git LFS fetched the file)
float_model = load_model(str(float_path))
print(f'input {float_model.input_shape} -> output {float_model.output_shape}, '
      f'{float_model.count_params():,} parameters\n')

padding = {l.name: l.padding[0] for l in float_model.layers if l.__class__.__name__ == 'ZeroPadding3D'}
for layer in float_model.layers:
    if layer.__class__.__name__ == 'Conv3D':
        print(f'{layer.name:16} kernel {str(layer.kernel_size):10} strides {str(layer.strides):10} '
              f'groups {layer.groups:4} -> {layer.output_shape}')
print('\nTemporal padding (before, after):', set(padding.values()))

input (None, 16, 100, 100, 3) -> output (None, 16, 1, 1, 27), 1,353,895 parameters

input_conv       kernel (1, 3, 3)  strides (1, 2, 2)  groups    1 -> (None, 16, 50, 50, 8)
convt_full_1_0   kernel (5, 1, 1)  strides (1, 1, 1)  groups    1 -> (None, 16, 50, 50, 20)
convs_full_1_0   kernel (1, 3, 3)  strides (1, 2, 2)  groups    1 -> (None, 16, 25, 25, 40)
convt_full_2_0   kernel (5, 1, 1)  strides (1, 1, 1)  groups    1 -> (None, 16, 25, 25, 80)
convs_full_2_0   kernel (1, 3, 3)  strides (1, 2, 2)  groups    1 -> (None, 16, 13, 13, 120)
convt_full_3_0   kernel (5, 1, 1)  strides (1, 1, 1)  groups    1 -> (None, 16, 13, 13, 160)
convs_full_3_0   kernel (1, 3, 3)  strides (1, 2, 2)  groups    1 -> (None, 16, 7, 7, 200)
convt_dw_4_0     kernel (5, 1, 1)  strides (1, 1, 1)  groups  200 -> (None, 16, 7, 7, 200)
convt_pw_4_0     kernel (1, 1, 1)  strides (1, 1, 1)  groups    1 -> (None, 16, 7, 7, 240)
convs_dw_4_0     kernel (1, 3, 3)  strides (1, 2, 2)  groups  240 -> (None, 16, 4, 4, 240)

## Data

Each clip becomes 16 frames, taken at a stride of 2 from its first frame and center-cropped
to 100 × 100, as raw uint8 pixels. There is no normalization: the model's first layer is a
`Rescaling(1.)` placeholder, and it was trained on pixels in [0, 255]. See
[jester_preprocessing.py](jester_preprocessing.py), whose bit-equality with the
`akida_models` pipeline `python jester_data.py --check-preprocessing` checks.

In [4]:
from jester_data import get_data, get_data_dir, read_labels

data_dir = get_data_dir(DATA_PATH)
dataset, num_clips = get_data(data_dir, BATCH_SIZE)
labels = read_labels(data_dir)
clips, clip_labels = next(iter(dataset))
print(f'{num_clips} clips, {len(labels)} classes')
print(f'batch: {clips.shape} {clips.dtype}, range [{clips.numpy().min()}, {clips.numpy().max()}]')
print('first clips:', [labels[i] for i in clip_labels.numpy()[:4]])

541 clips, 27 classes
batch: (32, 16, 100, 100, 3) <dtype: 'uint8'>, range [0, 255]
first clips: ['Thumb Up', 'Pushing Two Fingers Away', 'Swiping Left', 'Thumb Down']


## Float evaluation

The float model produces 16 per-frame predictions for each clip, and two accuracies come
from them, the same two for every variant in this notebook:

- **clip accuracy**: one decision per clip. The per-frame softmax outputs are accumulated
  with a decay of 0.8 (after each frame, the running total is multiplied by 0.8 and the new
  frame's softmax added), and the clip's class is the largest entry at the end. This is how
  BrainChip's `evaluate_bufferized_model` scores the model, and the headline figure.
- **frame accuracy**: each frame's own prediction against the clip's label. It is lower,
  because the first frames of a clip have seen little of the gesture.

In [5]:
from jester_eval import evaluate, float_frame_logits

float_result = evaluate(float_frame_logits(float_model), dataset, desc='Float')
print(f"\nFloat over {float_result['n']} clips: clip accuracy {float_result['clip_acc'] * 100:.2f}%, "
      f"frame accuracy {float_result['frame_acc'] * 100:.2f}%")

Float: 0it [00:00, ?it/s]I0000 00:00:1793058795.101650 2294261 cuda_dnn.cc:529] Loaded cuDNN version 90300
Float: 17it [00:14,  1.19it/s]


Float over 541 clips: clip accuracy 95.01%, frame accuracy 71.15%


## Quantization: 8-bit PTQ

BrainChip quantized this model with

```
quantizeml quantize -m tenn_spatiotemporal_jester.h5 -w 8 -i 8 -a 8 --per_tensor_activations -id uint8 -e 1 -bs 100 -sa jester_video_bs100.npz
```

`quantize_float_model` in [jester_model.py](jester_model.py) makes the same call through the
Python API. The calibration pack holds 100 train clips of 16 frames as one array of 1,600
frames in clip order. For a spatiotemporal TENN, `batch_size` must be the number of clips:
the calibration uses it to cut the frames back into clips, and `quantize()` refuses to run
without it.

Quantization also changes the model's structure. Each temporal `Conv3D` becomes a
`BufferTempConv` (`DepthwiseBufferTempConv` in the depthwise blocks), which keeps the last 5
frames' inputs in a FIFO buffer, and each spatial `Conv3D` becomes a 2D convolution. The
quantized model takes **one frame** and returns one prediction.

In [6]:
from collections import Counter
from jester_data import get_calibration_samples
from jester_model import quantize_float_model

calibration = get_calibration_samples()
print(f'calibration pack: {calibration.shape} {calibration.dtype}')
quantized_model = quantize_float_model(float_model, calibration, num_clips=100)

print(f'\ninput {quantized_model.input_shape} -> output {quantized_model.output_shape}')
print(Counter(l.__class__.__name__ for l in quantized_model.layers
              if 'Conv' in l.__class__.__name__))

calibration pack: (1600, 100, 100, 3) uint8
16/16 [==============================] - 3s 9ms/step

input (None, 100, 100, 3) -> output (None, 27)
Counter({'QuantizedConv2D': 8, 'QuantizedBufferTempConv': 3, 'QuantizedDepthwiseBufferTempConv': 2, 'QuantizedDepthwiseConv2D': 2})


### Comparison with the published model

`compare_models` checks the structure, the Keras variables, and the converted Akida models,
whose integer weights and thresholds are what runs on the NPs.

In [7]:
from jester_model import compare_models, print_comparison

published_path = model_path('quantized')
verify_model(published_path)
published_model = load_model(str(published_path))
print_comparison(compare_models(published_model, quantized_model))

Same structure (layers, shapes, variables): True
Keras variables that differ: 16
  convs_dw_4_0/depthwise_kernel:0                              max |diff| 4.77e-07
  convs_dw_4_0/relu/output_quantizer/rescaling_rate:0          max |diff| 5.96e-08
  convs_dw_5_0/depthwise_kernel:0                              max |diff| 1.91e-06
  convs_dw_5_0/relu/output_quantizer/rescaling_rate:0          max |diff| 5.96e-08
  convs_pw_4_0/kernel:0                                        max |diff| 2.38e-07
  convs_pw_4_0/relu/output_quantizer/rescaling_rate:0          max |diff| 5.96e-08
  convs_pw_5_0/kernel:0                                        max |diff| 1.19e-07
  convt_dw_4_0/relu/output_quantizer/rescaling_rate:0          max |diff| 1.19e-07
  convt_dw_5_0/kernel:0                                        max |diff| 9.54e-07
  convt_dw_5_0/relu/output_quantizer/rescaling_rate:0          max |diff| 5.96e-08
  convt_pw_4_0/kernel:0                                        max |diff| 9.54e-07
  conv

The structure and the integer weights are identical. The few differences are float
rounding, in Keras variables that are quantized afterwards and in the Akida model's float
output scales. The evaluation below uses the model just produced; the published one
evaluates identically.

## Streaming evaluation

The quantized model sees a clip one frame at a time, and its buffers carry state from
frame to frame. So the buffers are **reset at the start of every batch of clips**, with
`reset_buffers`, and the frames are fed in order. In Keras the batch axis holds independent
clips: a batch of 32 clips steps through its 16 frames together.

In [8]:
import inspect
from jester_eval import quantized_frame_logits

print(inspect.getsource(quantized_frame_logits))
quant_result = evaluate(quantized_frame_logits(quantized_model), dataset, desc='8-bit')
print(f"\n8-bit PTQ over {quant_result['n']} clips: clip accuracy {quant_result['clip_acc'] * 100:.2f}%, "
      f"frame accuracy {quant_result['frame_acc'] * 100:.2f}%")
print(f"Change vs float: {(quant_result['clip_acc'] - float_result['clip_acc']) * 100:+.2f} points (clip)")

def quantized_frame_logits(model):
    """Per-frame logits of the buffered Keras model, streaming each clip frame by frame."""
    model_fn = tf.function(model)

    def run(clips):
        reset_buffers(model)  # a new batch of clips: forget the previous frames
        return np.stack([model_fn(clips[:, t]).numpy() for t in range(clips.shape[1])], 1)
    return run



8-bit: 0it [00:00, ?it/s]W0000 00:00:1793058854.251893 2293786 loop_optimizer.cc:934] Skipping loop optimization for Merge node with control input: jester_video/convt_full_1_0/assert_less_equal/Assert/AssertGuard/branch_executed/_73
8-bit: 16it [00:13,  1.91it/s]W0000 00:00:1793058867.916671 2293786 loop_optimizer.cc:934] Skipping loop optimization for Merge node with control input: jester_video/convt_full_1_0/assert_less_equal/Assert/AssertGuard/branch_executed/_73
8-bit: 17it [00:19,  1.17s/it]


8-bit PTQ over 541 clips: clip accuracy 94.64%, frame accuracy 70.52%
Change vs float: -0.37 points (clip)


## Conversion to Akida

`cnn2snn.convert` turns the quantized Keras model into an Akida model. Conversion changes
the representation, not the computation, so the Akida model should score the same as the
quantized model.

In [9]:
from cnn2snn import convert

akida_model = convert(quantized_model)
akida_model.summary()

                 Model Summary                  
________________________________________________
Input shape    Output shape  Sequences  Layers
[100, 100, 3]  [1, 1, 27]    1          18    
________________________________________________

_________________________________________________________________________
Layer (type)                            Output shape   Kernel shape    

================== SW/input_conv-dequantizer (Software) =================

input_conv (InputConv2D)                [50, 50, 8]    (3, 3, 3, 8)    
_________________________________________________________________________
convt_full_1_0 (BufferTempConv)         [50, 50, 20]   (1, 1, 40, 20)  
_________________________________________________________________________
convs_full_1_0 (Conv2D)                 [25, 25, 40]   (3, 3, 20, 40)  
_________________________________________________________________________
convt_full_2_0 (BufferTempConv)         [25, 25, 80]   (1, 1, 200, 80) 
__________________________

### Streaming on Akida

An `akida.Model` has no buffer reset: a new model built from the same layers starts with
empty buffers. And on a buffered model, Akida reads the batch axis as **time**. Here one
clip is forwarded three ways:

In [10]:
import akida

clip = clips.numpy()[0]                                   # (16, 100, 100, 3)
one_by_one = akida.Model(layers=akida_model.layers)
stream = np.concatenate([one_by_one.forward(clip[t:t + 1]) for t in range(len(clip))])
batch = akida.Model(layers=akida_model.layers).forward(clip)
independent = np.concatenate([akida.Model(layers=akida_model.layers).forward(clip[t:t + 1])
                              for t in range(len(clip))])
print('16-frame batch == frame-by-frame stream:   ', np.array_equal(batch, stream))
print('16-frame batch == 16 independent frames:   ', np.array_equal(batch, independent))

16-frame batch == frame-by-frame stream:    True
16-frame batch == 16 independent frames:    False


So the Akida evaluation forwards one clip per call, on a fresh model. A batch of several
clips would be one long stream. This runs on the software backend and takes a couple of
minutes.

In [11]:
from jester_eval import akida_frame_logits

akida_dataset, _ = get_data(data_dir, batch_size=1)
akida_result = evaluate(akida_frame_logits(akida_model), akida_dataset, desc='Akida')
print(f"\nAkida over {akida_result['n']} clips: clip accuracy {akida_result['clip_acc'] * 100:.2f}%, "
      f"frame accuracy {akida_result['frame_acc'] * 100:.2f}%")

Akida: 541it [02:28,  3.63it/s]


Akida over 541 clips: clip accuracy 94.64%, frame accuracy 70.52%


## Mapping onto Akida 2 devices

Mapping is a software step, so you can check a model against a *virtual* device without the
hardware. `hw_only=True` makes the mapping fail rather than fall back to software for any
part of the model.

In [12]:
from brainchip_utils.hardware_utils import get_mapping_stats

devices = {'6-node FPGA': akida.SixNodesIPv2, '12-node': akida.TwelveNodesIPv2}
for name, make_device in devices.items():
    for mode in ['Minimal', 'AllNps', 'HwPr']:
        akida_model.map(make_device(), mode=getattr(akida.MapMode, mode), hw_only=True)
        nps, passes, sequences = get_mapping_stats(akida_model)
        print(f'{name:12} {mode:8} {nps:3d} NPs, {passes} pass(es), {sequences} sequence(s)')

6-node FPGA  Minimal   44 NPs, 2 pass(es), 1 sequence(s)
6-node FPGA  AllNps    45 NPs, 2 pass(es), 1 sequence(s)
6-node FPGA  HwPr      96 NPs, 5 pass(es), 1 sequence(s)
12-node      Minimal   44 NPs, 1 pass(es), 1 sequence(s)
12-node      AllNps    49 NPs, 1 pass(es), 1 sequence(s)
12-node      HwPr     184 NPs, 5 pass(es), 1 sequence(s)


On the FPGA, the model needs more than one pass even in Minimal mode, so the weights of
each pass are reloaded at every frame. The mapping summary also shows what is particular to
a temporal model: the buffered layers map onto `TNP_B` components, and each is allotted
external memory.

In [13]:
akida_model.map(akida.SixNodesIPv2(), mode=akida.MapMode.Minimal, hw_only=True)
akida_model.summary()

                                      Model Summary                                      
_________________________________________________________________________________________
Input shape    Output shape  Sequences  Layers  NPs  Skip DMAs  External Memory (Bytes)
[100, 100, 3]  [1, 1, 27]    1          18      43   5          892080                 
_________________________________________________________________________________________

_________________________
Component (type)  Count
HRC               1    
_________________________
CNP1              22   
_________________________
FNP2              1    
_________________________
FNP3              1    
_________________________
SKIP_DMA_STORE    5    
_________________________
TNP_B             19   
_________________________
SKIP_DMA_LOAD     5    
_________________________

                     External Memory Summary                     
_________________________________________________________________
Layer (type)        

## Activation sparsity

Akida is event-driven: a zero activation generates no event and so no work. The sparsity is
measured by streaming one clip of each class through the Akida model; `compute_sparsity`
builds fresh sub-models, so every clip starts from empty buffers.

In [14]:
from brainchip_utils.plot_utils import pretty_print_sparsity
from jester_eval import akida_sparsity

per_layer, mean_sparsity = akida_sparsity(akida_model, data_dir)
pretty_print_sparsity(per_layer)
print(f'\nMean activation sparsity: {mean_sparsity * 100:.2f}%')


Layer              Sparsity
---------------------------
input_conv          34.77%
convt_full_1_0      42.55%
convs_full_1_0      42.83%
convt_full_2_0      68.43%
convs_full_2_0      71.43%
convt_full_3_0      75.03%
convs_full_3_0      70.89%
convt_dw_4_0        60.03%
convt_pw_4_0        55.24%
convs_dw_4_0        47.18%
convs_pw_4_0        64.88%
convt_dw_5_0        47.56%
convt_pw_5_0        56.63%
convs_dw_5_0        38.70%
convs_pw_5_0        87.87%
dense               98.68%
dense_1              0.00%
---------------------------
Mean                56.63%

Mean activation sparsity: 56.63%


## Summary

Accuracy of each variant on the BrainChip Jester subset. The Akida model should match the
quantized model, since conversion changes the representation, not the computation; the
gap from float is the cost of 8-bit quantization.

In [15]:
print(f"BrainChip Jester subset, {akida_result['n']} clips (not the full validation split)")
for name, r in [('Float', float_result), ('8-bit PTQ', quant_result), ('Akida', akida_result)]:
    print(f"  {name:10} clip accuracy {r['clip_acc'] * 100:6.2f}%   frame accuracy {r['frame_acc'] * 100:6.2f}%")
print(f'  mean activation sparsity: {mean_sparsity * 100:.2f}%')

BrainChip Jester subset, 541 clips (not the full validation split)
  Float      clip accuracy  95.01%   frame accuracy  71.15%
  8-bit PTQ  clip accuracy  94.64%   frame accuracy  70.52%
  Akida      clip accuracy  94.64%   frame accuracy  70.52%
  mean activation sparsity: 56.63%


To measure the latency per frame on the Akida 2 FPGA, continue with
[jester_notebook_benchmark.ipynb](jester_notebook_benchmark.ipynb).